# Reasoning under uncertainty

The rules of an [expert system](rules_and_expert_systems.ipynb) are certain: if the conditions hold, the conclusion holds. Real evidence rarely works that way. A symptom makes a disease *more likely*, an alarm *suggests* a burglary. MYCIN attached *certainty factors* to its rules, numbers that combined in ways without a firm mathematical basis. The principled answer is **probability**: degrees of belief that are updated with **Bayes' theorem** as evidence arrives ([the theorem and a diagnostic test](../probability/README.md)).

The trouble is size. The full **joint distribution**, a probability for every combination of values, needs 2ⁿ − 1 numbers for *n* yes-or-no variables: about a million for 20 variables. A **Bayesian network** (Judea Pearl, 1985) avoids writing them all by recording only which variables directly influence which.

## A Bayesian network

Pearl's classic example. A house has a burglar alarm, which a burglary usually sets off, and a small earthquake sometimes does too. Two neighbours, John and Mary, have promised to call when they hear it; John sometimes mistakes the phone for the alarm, and Mary sometimes misses it over loud music.

```mermaid
flowchart TD
    burglary["Burglary"] --> alarm["Alarm"]
    earthquake["Earthquake"] --> alarm
    alarm --> john["John calls"]
    alarm --> mary["Mary calls"]
```

Each arrow means *directly influences*. Each variable gets a table of probabilities given its parents only: the calls depend on the alarm, not directly on the burglary. These are the numbers of Russell and Norvig's textbook, written by hand as an expert would:

In [ ]:
probability_of_burglary = 0.001
probability_of_earthquake = 0.002
probability_of_alarm = {  # given (burglary, earthquake)
    (True, True): 0.95, (True, False): 0.94, (False, True): 0.29, (False, False): 0.001,
}
probability_john_calls = {True: 0.90, False: 0.05}  # given alarm
probability_mary_calls = {True: 0.70, False: 0.01}  # given alarm

That is 10 numbers, where the full joint distribution of five yes-or-no variables needs 31. The joint probability of any complete situation is the product of each variable's probability given its parents:

In [ ]:
def probability_of(value, probability_of_true):
    return probability_of_true if value else 1 - probability_of_true


def joint_probability(burglary, earthquake, alarm, john_calls, mary_calls):
    return (probability_of(burglary, probability_of_burglary)
            * probability_of(earthquake, probability_of_earthquake)
            * probability_of(alarm, probability_of_alarm[(burglary, earthquake)])
            * probability_of(john_calls, probability_john_calls[alarm])
            * probability_of(mary_calls, probability_mary_calls[alarm]))


print(f"{joint_probability(False, False, True, True, True):.6f}",
      "= P(alarm, both call, but no burglary and no earthquake)")

0.000628 = P(alarm, both call, but no burglary and no earthquake)


## Inference by enumeration

To answer a question such as *how likely is a burglary, given that both neighbours called?*, add up the joint probability of every situation that agrees with the evidence, separately for each answer, and normalise. Like model checking in [logic](logic.ipynb), it enumerates every possible world, here weighting each one by its probability.

In [ ]:
from itertools import product

variables = ["burglary", "earthquake", "alarm", "john_calls", "mary_calls"]


def probability_given(query, evidence):
    totals = {True: 0.0, False: 0.0}
    for values in product([True, False], repeat=len(variables)):
        world = dict(zip(variables, values))
        if all(world[name] == value for name, value in evidence.items()):
            totals[world[query]] += joint_probability(*values)
    return totals[True] / (totals[True] + totals[False])

In [ ]:
both_call = {"john_calls": True, "mary_calls": True}
print(f"P(burglary | both call)   = {probability_given('burglary', both_call):.3f}")
print(f"P(earthquake | both call) = {probability_given('earthquake', both_call):.3f}")
print(f"P(burglary | only John is known to call) = {probability_given('burglary', {'john_calls': True}):.3f}")

P(burglary | both call)   = 0.284
P(earthquake | both call) = 0.176
P(burglary | only John is known to call) = 0.016


Even with both neighbours calling, a burglary is less likely than not: burglaries are so rare (one in a thousand) that a false alarm remains the more probable explanation. This is the same lesson as the [diagnostic test](../probability/diagnostic_test.md) of a rare disease: the **prior** matters as much as the evidence.

## Explaining away

Evidence for one cause can lower the probability of another. Once the alarm is known to have gone off, learning that there was an earthquake makes a burglary much less likely, because the earthquake already explains the alarm:

In [ ]:
print(f"P(burglary | alarm)             = {probability_given('burglary', {'alarm': True}):.4f}")
print(f"P(burglary | alarm, earthquake) = {probability_given('burglary', {'alarm': True, 'earthquake': True}):.4f}")

P(burglary | alarm)             = 0.3736
P(burglary | alarm, earthquake) = 0.0033


Rules can't express this: no if-then rule about burglaries mentions earthquakes. The network gets it for free from the structure of the influences, which is why Bayesian networks replaced certainty factors in diagnosis systems from the late 1980s.

Enumeration grows exponentially with the number of variables. Real systems use smarter exact algorithms, such as variable elimination, or approximate the answer by sampling.

## From knowledge to learning

The tables above were written by hand, as an expert would. They can also be **estimated from data**: simulate a hundred thousand nights in this neighbourhood, and count.

In [ ]:
import random

random.seed(0)
nights = []
for _ in range(100_000):
    burglary = random.random() < probability_of_burglary
    earthquake = random.random() < probability_of_earthquake
    alarm = random.random() < probability_of_alarm[(burglary, earthquake)]
    john_calls = random.random() < probability_john_calls[alarm]
    mary_calls = random.random() < probability_mary_calls[alarm]
    nights.append((burglary, earthquake, alarm, john_calls, mary_calls))

In [ ]:
nights_with_alarm = [night for night in nights if night[2]]
print("Nights with a burglary:", sum(night[0] for night in nights))
print("Nights with an alarm:", len(nights_with_alarm))
print(f"Estimated P(John calls | alarm) = {sum(night[3] for night in nights_with_alarm) / len(nights_with_alarm):.2f} (true 0.90)")
print(f"Estimated P(Mary calls | alarm) = {sum(night[4] for night in nights_with_alarm) / len(nights_with_alarm):.2f} (true 0.70)")
print("Nights with both a burglary and an earthquake:", sum(night[0] and night[1] for night in nights))

Nights with a burglary: 111
Nights with an alarm: 258
Estimated P(John calls | alarm) = 0.87 (true 0.90)
Estimated P(Mary calls | alarm) = 0.68 (true 0.70)
Nights with both a burglary and an earthquake: 0


Counting recovers the frequent probabilities, close to the true ones. The rare combinations are another matter: a burglary during an earthquake should happen about twice in a million nights, and in these hundred thousand it never did, so its row of the alarm's table can't be estimated at all. Where data is scarce, an expert's numbers fill the gap; where it is plentiful, counting is cheaper and more accurate than asking.

Estimating the tables from data is **machine learning**: the network's structure written by a person, its probabilities learned. The **naive Bayes** classifier is the simplest case, a network in which the class points to every feature, with every probability counted from labelled examples; it is a deep dive of supervised learning *(coming)*. So Bayesian reasoning is not on one side of the line between classical AI and machine learning: what decides the side is where the probabilities come from.